In [ ]:
import sys

sys.path.append("../")

from method.watermarking import *
from method.diffusion import *




In [ ]:
import torch

entity_embeddings = torch.load("../datasets/ali/embeddings.pt")
entity_embeddings.shape # should be (num_entities, embedding_dim)

## Embed Watermark

In [ ]:
batch_size = 10

ddim_scheduler = SimpleDDIMScheduler(batch_size)

x = entity_embeddings[:batch_size] # take the first batch_size entity embeddings as input

inversed_latents = ddim_scheduler.inverse(x)
inversed_latents_all = ddim_scheduler.inverse_all_step(x)
sampled_latents_all = ddim_scheduler.sample_all_step(inversed_latents)

sig = get_watermarking_signature(batch_size)
mask = get_watermarking_mask(
    inversed_latents_all,
    sampled_latents_all,
    sig,
    [5, 15, 25, 35, 45],
)

watermarked_inv_latents = embed_watermark(inversed_latents, mask, sig)

watermarked_x = ddim_scheduler.sample(watermarked_inv_latents)

watermarked_x.shape # should be (batch_size, embedding_dim)

## Detect Watermark

In [ ]:
target_inv_latents = ddim_scheduler.inverse(watermarked_x)

p = get_p_value(target_inv_latents, mask, sig)

print(f"p-value: {p}")